
# BIAA · Rasterització LabelMe → PNG (0..9 / 255) + Split 70/15/15

Notebook complet per:
1) Construir **split 70/15/15** a partir de carpetes planes (`images_all` + `labelme_all` o `segmaps_all`).  
2) **Diagnosticar etiquetes** a LabelMe (detecta dígits `"0..9, 255"` o noms).  
3) **Rasteritzar LabelMe JSON → PNG index** (0..9; 255=ignore) per a `train/val/test`.  
4) **Verificar** que les màscares continguin valors esperats.

> Dependències: `opencv-python`, `numpy`, `tqdm`.


## 1) Setup

In [2]:

#@title Instal·la deps (si cal) i comprova versions
!pip -q install opencv-python numpy tqdm

import sys, os, json, re, unicodedata, random, shutil
from pathlib import Path
import cv2, numpy as np
from tqdm import tqdm
import collections

print("OK: OpenCV", cv2.__version__)


OK: OpenCV 4.12.0


## 2) Configuració bàsica

In [10]:

#@title Paràmetres de configuració (edita segons el teu entorn)
class CFG:
    data_root = Path("/home/joan_ds/Sandbox/UOC/TFM/data/dataset_500_GT")
    mode = "labelme"  # "labelme" o "png"

    src_images_flat  = data_root / "images"
    src_labelme_flat = data_root / "images/annotations"
    src_segmap_flat  = data_root / "segmaps"

    dst_images_dir = data_root / "images"
    dst_ann_dir    = data_root / ("labelme" if mode=="labelme" else "segmaps")

    splits = ("train","val","test")
    split_ratio = (0.70, 0.15, 0.15)

    out_segmaps_dir = data_root / "segmaps"
    overwrite_png   = True
    verify_masks    = True

CFG = CFG()
print("data_root:", CFG.data_root)
print("mode:", CFG.mode)


data_root: /home/joan_ds/Sandbox/UOC/TFM/data/dataset_500_GT
mode: labelme


## 3) Mapeig d’etiquetes i prioritat

In [4]:

#@title Defineix classes, mapeig numèric i sinònims (edita si cal)
CLASS_TO_INDEX = {
    "sidewalk_tiles": 0,
    "sidewalk_asphalt": 1,
    "roadway": 2,
    "curb_edge": 3,
    "drainage_inlet": 4,
    "gutter": 5,
    "access_cover": 6,
    "tree_pit": 7,
    "vegetation": 8,
    "street_furniture": 9,
}
NUMERIC_LABEL_MAP = {
    "0": "sidewalk_tiles",
    "1": "sidewalk_asphalt",
    "2": "roadway",
    "3": "curb_edge",
    "4": "drainage_inlet",
    "5": "gutter",
    "6": "access_cover",
    "7": "tree_pit",
    "8": "vegetation",
    "9": "street_furniture",
    "255": "ignore",
}
LABEL_SYNONYMS = {
    "tiles": "sidewalk_tiles",
    "panot": "sidewalk_tiles",
    "asphalt_sidewalk": "sidewalk_asphalt",
    "asphalt": "sidewalk_asphalt",
    "road": "roadway", "street": "roadway",
    "curb": "curb_edge", "kerb": "curb_edge",
    "drain": "drainage_inlet", "gully": "drainage_inlet", "inlet": "drainage_inlet", "grate": "drainage_inlet",
    "rigola": "gutter",
    "accesscover": "access_cover", "manhole": "access_cover", "manhole_cover": "access_cover",
    "treepit": "tree_pit",
    "bush": "vegetation", "tree": "vegetation",
    "furniture": "street_furniture", "street_forniture": "street_furniture",
}
PRIORITY = [
    "drainage_inlet",
    "access_cover",
    "curb_edge",
    "gutter",
    "street_furniture",
    "tree_pit",
    "vegetation",
    "sidewalk_tiles",
    "sidewalk_asphalt",
    "roadway",
]
IGNORE_VAL = 255


## 4) Utilitats (normalització i dibuix)

In [5]:

def strip_accents(s: str) -> str:
    return "".join(c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn")

def canonical_label(raw) -> str:
    s = str(raw).strip()
    if s.isdigit() and s in NUMERIC_LABEL_MAP:
        return NUMERIC_LABEL_MAP[s]
    s = strip_accents(s).lower()
    s = s.replace("-", "_").replace(" ", "_")
    s = re.sub(r"\(.*?\)$", "", s).strip("_")
    return LABEL_SYNONYMS.get(s, s)

def draw_polygons(mask: np.ndarray, polygons: list, value: int):
    if not polygons: return
    cnts = []
    for poly in polygons:
        if len(poly) < 3: continue
        arr = np.asarray(poly, dtype=np.float32)
        arr = np.round(arr).astype(np.int32)
        cnts.append(arr.reshape(-1, 1, 2))
    if cnts:
        cv2.fillPoly(mask, cnts, color=int(value))

def draw_rectangle(mask: np.ndarray, p0, p1, value: int):
    x0,y0 = p0; x1,y1 = p1
    poly = [[x0,y0],[x1,y0],[x1,y1],[x0,y1]]
    draw_polygons(mask, [poly], value)


## 5) JSON → Màscara d’índex i resolució d’imatge

In [6]:

def labelme_json_to_index_mask(data: dict, H: int, W: int) -> np.ndarray:
    h = int(data.get("imageHeight") or 0) or H
    w = int(data.get("imageWidth")  or 0) or W
    mask = np.full((h, w), IGNORE_VAL, dtype=np.uint8)

    by_label = {lab: [] for lab in PRIORITY}
    for shp in data.get("shapes", []):
        lab = canonical_label(shp.get("label"))
        if lab == "ignore":
            continue
        if lab not in CLASS_TO_INDEX:
            continue
        st = shp.get("shape_type", "polygon")
        pts = shp.get("points", [])
        if st == "polygon" and len(pts) >= 3:
            by_label[lab].append(pts)
        elif st == "rectangle" and len(pts) >= 2:
            (x0,y0),(x1,y1) = pts[0], pts[1]
            by_label[lab].append([[x0,y0],[x1,y0],[x1,y1],[x0,y1]])

    for lab in PRIORITY:
        polys = by_label.get(lab, [])
        if polys:
            draw_polygons(mask, polys, CLASS_TO_INDEX[lab])
    return mask

def load_image_for_json(images_dir: Path, json_path: Path):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if data.get("imagePath"):
        img_name = Path(data["imagePath"]).name
        cand = images_dir / img_name
        if cand.exists():
            return cand, data
    for ext in (".jpg",".jpeg",".png",".bmp"):
        cand = images_dir / f"{json_path.stem}{ext}"
        if cand.exists():
            return cand, data
    return None, data


## 6) Construir split 70/15/15 (si cal)

In [11]:
from pathlib import Path
import json

print("MODE:", CFG.mode)
print("src_images_flat :", CFG.src_images_flat,  "→", len(list(CFG.src_images_flat.glob("*"))), "fitxers")
if CFG.mode == "labelme":
    print("src_labelme_flat:", CFG.src_labelme_flat, "→", len(list(CFG.src_labelme_flat.glob("*.json"))), "jsons")
else:
    print("src_segmap_flat :", CFG.src_segmap_flat, "→", len(list(CFG.src_segmap_flat.glob("*.png"))), "pngs")

# Mirem un exemple si n'hi ha
if CFG.mode == "labelme":
    some_jsons = list(CFG.src_labelme_flat.glob("*.json"))[:3]
    print("\nExemples de JSON:")
    for jp in some_jsons:
        with open(jp, "r", encoding="utf-8") as f:
            data = json.load(f)
        print(" -", jp.name, "imagePath:", data.get("imagePath"))
else:
    print("\nExemples d'imatge a src_images_flat:")
    for p in list(CFG.src_images_flat.glob("*"))[:3]:
        print(" -", p.name)

MODE: labelme
src_images_flat : /home/joan_ds/Sandbox/UOC/TFM/data/dataset_500_GT/images → 513 fitxers
src_labelme_flat: /home/joan_ds/Sandbox/UOC/TFM/data/dataset_500_GT/images/annotations → 507 jsons

Exemples de JSON:
 - 2023-08-18-07-27-12.json imagePath: ../2023-08-18-07-27-12.jpg
 - 2025-04-19-21-28-12.json imagePath: ../2025-04-19-21-28-12.jpg
 - 2023-08-18-07-20-14.json imagePath: ../images/2023-08-18-07-20-14.jpg


In [12]:

def build_split_from_flat(cfg):
    if cfg.mode == "labelme":
        have_flat = cfg.src_images_flat.exists() and cfg.src_labelme_flat.exists()
    else:
        have_flat = cfg.src_images_flat.exists() and cfg.src_segmap_flat.exists()

    if not have_flat:
        print("No detecto carpetes *all poblades; assumint que els splits ja existeixen.")
        return

    for split in cfg.splits:
        (cfg.dst_images_dir/split).mkdir(parents=True, exist_ok=True)
        (cfg.dst_ann_dir/split).mkdir(parents=True, exist_ok=True)

    img_exts = {".jpg",".jpeg",".png",".bmp"}
    all_imgs = sorted([p for p in cfg.src_images_flat.glob("*") if p.suffix.lower() in img_exts])

    pairs = []
    if cfg.mode == "labelme":
        all_jsons = sorted(cfg.src_labelme_flat.glob("*.json"))
        def find_image_for_json(jp: Path) -> Path|None:
            try:
                with open(jp, "r", encoding="utf-8") as f:
                    data = json.load(f)
                if data.get("imagePath"):
                    name = Path(data["imagePath"]).name
                    cand = cfg.src_images_flat / name
                    if cand.exists():
                        return cand
            except Exception:
                pass
            stem = jp.stem
            for ext in img_exts:
                cand = cfg.src_images_flat / f"{stem}{ext}"
                if cand.exists():
                    return cand
            return None
        for jp in all_jsons:
            ip = find_image_for_json(jp)
            if ip is not None:
                pairs.append((ip, jp))
    else:
        pngs = {p.stem: p for p in cfg.src_segmap_flat.glob("*.png")}
        imgs = {p.stem: p for p in all_imgs}
        common = sorted(set(pngs.keys()) & set(imgs.keys()))
        for k in common:
            pairs.append((imgs[k], pngs[k]))

    print(f"Parelles trobades: {len(pairs)}")
    assert len(pairs) > 0, "No s'han trobat parelles. Revisa rutes *_all i mode."

    random.seed(42); random.shuffle(pairs)
    n = len(pairs)
    n_tr = int(cfg.split_ratio[0]*n)
    n_va = int(cfg.split_ratio[1]*n)
    train_pairs = pairs[:n_tr]
    val_pairs   = pairs[n_tr:n_tr+n_va]
    test_pairs  = pairs[n_tr+n_va:]

    print(f"Split → Train:{len(train_pairs)}  Val:{len(val_pairs)}  Test:{len(test_pairs)}")

    def copy_pair(ip: Path, ap: Path, split: str):
        dst_img = cfg.dst_images_dir/split/ip.name
        dst_ann = cfg.dst_ann_dir/split/(ap.name if cfg.mode=="labelme" else f"{ip.stem}.png")
        if not dst_img.exists():
            shutil.copy2(ip, dst_img)
        if not dst_ann.exists():
            shutil.copy2(ap, dst_ann)

    for (ip, ap) in train_pairs: copy_pair(ip, ap, "train")
    for (ip, ap) in val_pairs:   copy_pair(ip, ap, "val")
    for (ip, ap) in test_pairs:  copy_pair(ip, ap, "test")

    print("✔ Split creat a:", cfg.dst_images_dir, "i", cfg.dst_ann_dir)

build_split_from_flat(CFG)


Parelles trobades: 507
Split → Train:354  Val:76  Test:77
✔ Split creat a: /home/joan_ds/Sandbox/UOC/TFM/data/dataset_500_GT/images i /home/joan_ds/Sandbox/UOC/TFM/data/dataset_500_GT/labelme


## 7) Diagnòstic d’etiquetes

In [13]:

def diagnose_labels(cfg, limit_per_split=5000):
    if cfg.mode != "labelme":
        print("Diagnòstic d’etiquetes: saltant (mode=png).")
        return
    print("Classes esperades:", sorted(CLASS_TO_INDEX.keys()))
    print("Sinònims:", sorted(LABEL_SYNONYMS.keys())[:20], "...")

    for split in cfg.splits:
        json_dir = cfg.data_root/"labelme"/split
        if not json_dir.exists():
            print(f"[{split}] sense dir {json_dir} (saltant)")
            continue
        jsons = sorted(json_dir.glob("*.json"))
        lab_counter = collections.Counter()
        raw_samples = collections.Counter()
        for jp in jsons[:limit_per_split]:
            try:
                with open(jp, "r", encoding="utf-8") as f:
                    data = json.load(f)
            except Exception as e:
                print(f"[{split}] JSON KO {jp.name}: {e}")
                continue
            for shp in data.get("shapes", []):
                raw = str(shp.get("label", ""))
                raw_samples[raw] += 1
                lab = canonical_label(raw)
                lab_counter[lab] += 1

        valid = {k:v for k,v in lab_counter.items() if k in CLASS_TO_INDEX}
        invalid = {k:v for k,v in lab_counter.items() if (k not in CLASS_TO_INDEX and k != "ignore")}

        print(f"\n[{split}] JSONs: {len(jsons)} | etiquetes totals: {sum(lab_counter.values())}")
        print(f"  ✔ vàlides: {len(valid)} tipus, {sum(valid.values())} shapes")
        print(f"  ✖ NO mapegen (excloent 'ignore'): {len(invalid)} tipus, {sum(invalid.values())} shapes")
        print("  Top 10 NO mapegen:", sorted(invalid.items(), key=lambda x:-x[1])[:10])
        print("  Mostres raw:", raw_samples.most_common(10))

diagnose_labels(CFG)


Classes esperades: ['access_cover', 'curb_edge', 'drainage_inlet', 'gutter', 'roadway', 'sidewalk_asphalt', 'sidewalk_tiles', 'street_furniture', 'tree_pit', 'vegetation']
Sinònims: ['accesscover', 'asphalt', 'asphalt_sidewalk', 'bush', 'curb', 'drain', 'furniture', 'grate', 'gully', 'inlet', 'kerb', 'manhole', 'manhole_cover', 'panot', 'rigola', 'road', 'street', 'street_forniture', 'tiles', 'tree'] ...

[train] JSONs: 354 | etiquetes totals: 3707
  ✔ vàlides: 10 tipus, 3126 shapes
  ✖ NO mapegen (excloent 'ignore'): 0 tipus, 0 shapes
  Top 10 NO mapegen: []
  Mostres raw: [('0', 766), ('255', 581), ('6', 488), ('9', 405), ('3', 355), ('5', 302), ('2', 253), ('7', 227), ('8', 195), ('4', 93)]

[val] JSONs: 76 | etiquetes totals: 828
  ✔ vàlides: 10 tipus, 702 shapes
  ✖ NO mapegen (excloent 'ignore'): 0 tipus, 0 shapes
  Top 10 NO mapegen: []
  Mostres raw: [('0', 149), ('255', 126), ('6', 113), ('9', 109), ('3', 80), ('5', 61), ('2', 56), ('7', 54), ('8', 49), ('4', 19)]

[test] JSON

## 8) Rasterització LabelMe → PNG

In [14]:

def rasterize_labelme_splits(cfg):
    if cfg.mode != "labelme":
        print("mode != 'labelme' (saltant rasterització).")
        return
    total_written = 0
    for split in cfg.splits:
        images_dir = cfg.data_root / "images" / split
        json_dir   = cfg.data_root / "labelme" / split
        out_dir    = cfg.out_segmaps_dir / split
        out_dir.mkdir(parents=True, exist_ok=True)

        json_files = sorted(json_dir.glob("*.json"))
        if not json_files:
            print(f"[WARN] Sense JSONs a {json_dir} — saltant {split}.")
            continue

        print(f"[{split}] Rasteritzant {len(json_files)} JSONs → {out_dir}")
        blanks = 0
        for jp in tqdm(json_files):
            out_png = out_dir / f"{jp.stem}.png"
            if out_png.exists() and not cfg.overwrite_png:
                continue

            ip, data = load_image_for_json(images_dir, jp)
            H = W = None
            if ip and ip.exists():
                img = cv2.imread(str(ip), cv2.IMREAD_COLOR)
                if img is not None:
                    H, W = img.shape[:2]

            mask = labelme_json_to_index_mask(data, H or 0, W or 0)
            cv2.imwrite(str(out_png), mask)
            total_written += 1
            if np.all(mask == IGNORE_VAL):
                blanks += 1
        if blanks:
            print(f"  ⚠ {blanks} màscares tot 255 (cap shape vàlid). Revisa mapeig numèric/sinònims/priority.")

    print(f"✔ PNG escrits: {total_written}")

rasterize_labelme_splits(CFG)


[train] Rasteritzant 354 JSONs → /home/joan_ds/Sandbox/UOC/TFM/data/dataset_500_GT/segmaps/train


100%|██████████| 354/354 [00:10<00:00, 33.81it/s] 


[val] Rasteritzant 76 JSONs → /home/joan_ds/Sandbox/UOC/TFM/data/dataset_500_GT/segmaps/val


100%|██████████| 76/76 [00:02<00:00, 34.33it/s] 


[test] Rasteritzant 77 JSONs → /home/joan_ds/Sandbox/UOC/TFM/data/dataset_500_GT/segmaps/test


100%|██████████| 77/77 [00:02<00:00, 36.95it/s] 

✔ PNG escrits: 507


## 9) Verificació de màscares

In [15]:

def verify_some_masks(cfg, split="train", max_show=5):
    segdir = cfg.out_segmaps_dir if cfg.mode=="labelme" else cfg.dst_ann_dir
    segdir = segdir / split
    if not segdir.exists():
        print("No hi ha segmaps per verificar a:", segdir)
        return
    files = list(segdir.glob("*.png"))
    if not files:
        print("Sense PNG a verificar a:", segdir)
        return
    import random
    random.shuffle(files)
    for fp in files[:max_show]:
        m = cv2.imread(str(fp), cv2.IMREAD_UNCHANGED)
        if m is None:
            print("No puc llegir:", fp); continue
        vals = np.unique(m)
        print(f"{fp.name}: únics={vals[:30]} ...  tot255? {np.all(m==255)}")

if CFG.verify_masks:
    verify_some_masks(CFG, split="train", max_show=5)


2023-06-30-19-21-23.png: únics=[  0   1   2   9 255] ...  tot255? False
2023-07-05-07-46-58.png: únics=[  0   2   3   8   9 255] ...  tot255? False
2025-04-06-20-39-57.png: únics=[  0   2   3   4 255] ...  tot255? False
2025-03-01-18-06-51.png: únics=[  0   3   7   8 255] ...  tot255? False
2023-08-18-07-21-15.png: únics=[  0   2   3   7   8 255] ...  tot255? False
